
# Advanced Urban Mobility Visualization Suite

- Hidden demand city
- Directional mismatch corridor
- Urban ecology
- Temporal identity shift
- Robust hidden station atlas

Outputs:
- Interactive HTML
- PNG figures
- Satellite-style urban maps


In [1]:

# ============================================================
# SAFE INSTALL
# ============================================================

import sys
import subprocess

packages = [
    "plotly",
    "folium",
    "pydeck",
    "kaleido",
    "matplotlib",
    "pandas",
    "numpy"
]

for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p])


In [2]:

# ============================================================
# IMPORT
# ============================================================

import os
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import pydeck as pdk

import folium
from folium.plugins import HeatMap

import matplotlib.pyplot as plt

# ============================================================
# PATH
# ============================================================

BASE_DIR = r"C:\Users\82108\Desktop\새 폴더 (8)"

INPUT_PATH = os.path.join(
    BASE_DIR,
    "diverse_mobility_subway_analysis_filtered",
    "08_compare_subway_total_filtered.csv"
)

MAPPING_PATH = os.path.join(
    BASE_DIR,
    "station_mapping_final",
    "cell_to_nearest_station_VALID_5km_MASTER_ID_FILTERED.csv"
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "FINAL_URBAN_VISUALS"
)

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD
# ============================================================

print("[READ DF]")
df = pd.read_csv(INPUT_PATH)

print(df.shape)
print(df.columns.tolist())

print("[READ MAPPING]")
mapping = pd.read_csv(MAPPING_PATH)

# ============================================================
# COORD
# ============================================================

coord = (
    mapping[
        [
            "MASTER_STATION_ID",
            "station_lat",
            "station_lon"
        ]
    ]
    .drop_duplicates()
)

df = df.merge(
    coord,
    on="MASTER_STATION_ID",
    how="left"
)

print("[MISSING COORD]")
print(df["station_lat"].isna().sum())

# ============================================================
# CENTER
# ============================================================

CENTER_LAT = 37.55
CENTER_LON = 126.99

# ============================================================
# SAVE HELPER
# ============================================================

def save_plotly(fig, name):

    html_path = os.path.join(SAVE_DIR, f"{name}.html")
    png_path = os.path.join(SAVE_DIR, f"{name}.png")

    fig.write_html(html_path)

    try:
        fig.write_image(png_path, scale=3)
    except Exception as e:
        print("[PNG SAVE FAILED]", name, e)

    print("[SAVED]", name)

# ============================================================
# 1. HIDDEN CITY
# ============================================================

fig = px.scatter_mapbox(
    df,
    lat="station_lat",
    lon="station_lon",
    color="hidden_demand_score",
    size="hidden_demand_score",
    hover_name="nearest_station",
    hover_data=[
        "nearest_line",
        "hidden_demand_score"
    ],
    color_continuous_scale="Turbo",
    zoom=10,
    height=900
)

fig.update_layout(
    mapbox_style="carto-darkmatter",
    title="Hidden Urban Demand"
)

save_plotly(fig, "01_hidden_city")

# ============================================================
# 2. DIRECTIONAL CONFLICT
# ============================================================

top_mismatch = df.sort_values(
    "directional_mismatch_score",
    ascending=False
).head(80)

fig = px.scatter_mapbox(
    top_mismatch,
    lat="station_lat",
    lon="station_lon",
    color="directional_mismatch_score",
    size="directional_mismatch_score",
    hover_name="nearest_station",
    color_continuous_scale="Reds",
    zoom=10,
    height=900
)

fig.update_layout(
    mapbox_style="carto-darkmatter",
    title="Directional Conflict Corridor"
)

save_plotly(fig, "02_directional_conflict")

# ============================================================
# 3. OBSERVED VS HIDDEN
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Scattermapbox(
        lat=df["station_lat"],
        lon=df["station_lon"],
        mode="markers",
        marker=dict(
            size=df["subway_norm"] * 25,
            color="cyan",
            opacity=0.35
        ),
        name="Observed Subway"
    )
)

fig.add_trace(
    go.Scattermapbox(
        lat=df["station_lat"],
        lon=df["station_lon"],
        mode="markers",
        marker=dict(
            size=df["mobility_norm"] * 25,
            color="magenta",
            opacity=0.35
        ),
        name="Actual Mobility"
    )
)

fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_zoom=10,
    mapbox_center={"lat": CENTER_LAT, "lon": CENTER_LON},
    height=900,
    title="Observed vs Hidden City"
)

save_plotly(fig, "03_observed_vs_hidden")

# ============================================================
# 4. STATION ECOLOGY
# ============================================================

fig = px.scatter_mapbox(
    df,
    lat="station_lat",
    lon="station_lon",
    color="station_type",
    size="hidden_demand_score",
    hover_name="nearest_station",
    zoom=10,
    height=900
)

fig.update_layout(
    mapbox_style="carto-darkmatter",
    title="Urban Station Ecology"
)

save_plotly(fig, "04_station_ecology")

# ============================================================
# 5. TEMPORAL IDENTITY
# ============================================================

fig = px.scatter_mapbox(
    df,
    lat="station_lat",
    lon="station_lon",
    color="dominant_period",
    size="surrounding_total_flow",
    hover_name="nearest_station",
    zoom=10,
    height=900
)

fig.update_layout(
    mapbox_style="satellite-streets",
    title="Temporal Identity Shift"
)

save_plotly(fig, "05_temporal_identity")

# ============================================================
# 6. ROBUST HIDDEN
# ============================================================

robust = df.sort_values(
    "hidden_demand_score",
    ascending=False
).head(32)

fig = px.scatter_mapbox(
    robust,
    lat="station_lat",
    lon="station_lon",
    size="hidden_demand_score",
    color="hidden_demand_score",
    hover_name="nearest_station",
    zoom=10,
    height=900
)

fig.update_layout(
    mapbox_style="satellite-streets",
    title="Top-32 Robust Hidden Stations"
)

save_plotly(fig, "06_robust_hidden")

# ============================================================
# 7. FOLIUM HEATMAP
# ============================================================

m = folium.Map(
    location=[CENTER_LAT, CENTER_LON],
    zoom_start=10,
    tiles="CartoDB dark_matter"
)

heat_data = []

for _, row in df.iterrows():

    heat_data.append([
        row["station_lat"],
        row["station_lon"],
        row["hidden_demand_score"]
    ])

HeatMap(
    heat_data,
    radius=25,
    blur=20
).add_to(m)

m.save(
    os.path.join(
        SAVE_DIR,
        "07_hidden_heatmap.html"
    )
)

print("[SAVED] folium heatmap")

# ============================================================
# 8. PYDECK 3D CITY
# ============================================================

deck_df = df.copy()

deck = pdk.Deck(
    map_style="mapbox://styles/mapbox/dark-v10",
    initial_view_state=pdk.ViewState(
        latitude=CENTER_LAT,
        longitude=CENTER_LON,
        zoom=10,
        pitch=50,
    ),
    layers=[
        pdk.Layer(
            "ColumnLayer",
            data=deck_df,
            get_position='[station_lon, station_lat]',
            get_elevation='hidden_demand_score * 50000',
            elevation_scale=1,
            radius=700,
            get_fill_color='[255, 80, 120, 180]',
            pickable=True,
            auto_highlight=True,
        )
    ],
    tooltip={
        "text": "{nearest_station}\nHidden: {hidden_demand_score}"
    }
)

deck.to_html(
    os.path.join(
        SAVE_DIR,
        "08_pydeck_hidden_city.html"
    )
)

print("[SAVED] pydeck")

# ============================================================
# 9. STATIC PAPER FIGURE
# ============================================================

top20 = df.sort_values(
    "hidden_demand_score",
    ascending=False
).head(20)

plt.figure(figsize=(16,10))

plt.scatter(
    top20["hidden_demand_score"],
    top20["diverse_living_area_score"],
    s=top20["surrounding_total_flow"] / 5e6,
    alpha=0.7
)

for _, row in top20.iterrows():

    plt.text(
        row["hidden_demand_score"],
        row["diverse_living_area_score"],
        row["nearest_station"],
        fontsize=9
    )

plt.xlabel("Hidden Demand")
plt.ylabel("Diverse Living Area")
plt.title("Latent Urban Activity Landscape")

plt.savefig(
    os.path.join(
        SAVE_DIR,
        "09_latent_urban_landscape.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("[SAVED] paper figure")

print()
print("DONE")
print(SAVE_DIR)


[READ DF]
(327, 47)
['MASTER_STATION_ID', 'nearest_station', 'nearest_line', 'mapped_cell_count', 'active_cell_count', 'surrounding_total_flow', 'surrounding_outflow', 'surrounding_inflow', 'surrounding_netflow', 'mean_cell_flow', 'median_cell_flow', 'max_cell_flow', 'mean_distance_m', 'min_distance_m', 'max_distance_m', 'mean_directional_imbalance', 'morning_peak_flow', 'daytime_flow', 'evening_peak_flow', 'night_flow', 'mean_temporal_entropy', 'active_cell_ratio', 'station_abs_net_flow', 'station_directional_imbalance', 'dominant_period', 'period_entropy', 'distance_band_entropy', 'outer_1km_plus_share', 'subway_out_sum', 'subway_in_sum', 'subway_net_sum', 'subway_record_count', 'subway_file_count', 'subway_total_activity', 'subway_abs_net_flow', 'subway_directional_imbalance', 'mobility_norm', 'subway_norm', 'mobility_minus_subway_gap', 'hidden_demand_score', 'subway_overrepresentation_score', 'distance_norm', 'distance_weighted_hidden_demand', 'outer_hidden_demand_score', 'diverse_

c:\Users\82108\anaconda3\dfcjsic\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




[PNG SAVE FAILED] 01_hidden_city 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

[SAVED] 01_hidden_city
[PNG SAVE FAILED] 02_directional_conflict 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

[SAVED] 02_directional_conflict
[PNG SAVE FAILED] 03_observed_vs_hidden 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

[SAVED] 03_observed_vs_hidden
[PNG SAVE FAILED] 04_station_ecology 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

[SAVED] 04_station_ecology
[PNG SAVE FAILED] 05_temporal_identity 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

[SAVED] 05_temporal_id

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30320\2990092796.py:383: UserWarning:

Glyph 44620 (\N{HANGUL SYLLABLE GGA}) missing from font(s) DejaVu Sans.

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30320\2990092796.py:383: UserWarning:

Glyph 52824 (\N{HANGUL SYLLABLE CI}) missing from font(s) DejaVu Sans.

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30320\2990092796.py:383: UserWarning:

Glyph 49328 (\N{HANGUL SYLLABLE SAN}) missing from font(s) DejaVu Sans.

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30320\2990092796.py:383: UserWarning:

Glyph 45800 (\N{HANGUL SYLLABLE DAN}) missing from font(s) DejaVu Sans.

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30320\2990092796.py:383: UserWarning:

Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30320\2990092796.py:383: UserWarning:

Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sa

[SAVED] paper figure

DONE
C:\Users\82108\Desktop\새 폴더 (8)\FINAL_URBAN_VISUALS
